# CareImpact AI: fine-tune Qwen2.5-1.5B on a free Colab GPU

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.** When asked, upload `careimpact_bundle.zip` (from the project's `colab` folder) and allow Google Drive access.
3. Wait about 1–2 hours. The result is saved to **Google Drive → careimpact → careimpact_ft.zip** (about 1.7 GB).
4. Download that zip to your laptop's **Downloads** folder and run `colab\import_finetuned.ps1` in the project.

If Colab disconnects, run all again: the base model is re-downloaded and training restarts.

In [ ]:
# 1. Check the GPU
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then Run all again.'
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Upload careimpact_bundle.zip
import os, zipfile
from google.colab import files
os.makedirs('/content/careimpact', exist_ok=True)
%cd /content/careimpact
if not os.path.exists('careimpact/training/train_lora.py'):
    up = files.upload()
    zipfile.ZipFile(next(iter(up))).extractall('.')
!ls data/training/final && wc -l data/training/final/*.jsonl

In [ ]:
# 3. Install training libraries and get the GGUF converter (uses Colab's preinstalled torch)
!pip install -q -U "transformers>=4.56" "peft>=0.13" accelerate safetensors sentencepiece pyyaml psutil
!test -d /content/llama.cpp || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp

In [ ]:
# 4. Train (LoRA, 2 epochs over the training set). Progress prints one line per step.
!python -m careimpact.training.train_lora --config configs/training/colab_t4_1p5b.yaml

In [ ]:
# 5. Merge the adapter and convert to GGUF for Ollama
!python -m careimpact.training.export --run models/runs/colab_1p5b_lora --name careimpact-ft --llama-cpp /content/llama.cpp
!cat models/runs/colab_1p5b_lora/summary.json | head -40

In [ ]:
# 6. Save the result to Google Drive (more reliable than a browser download for large files)
import shutil
from google.colab import drive
drive.mount('/content/drive')
run = 'models/runs/colab_1p5b_lora'
out = '/content/careimpact_ft.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_STORED) as z:
    for name in ['careimpact-ft.gguf', 'Modelfile', 'summary.json', 'metrics.jsonl', 'config.yaml']:
        z.write(f'{run}/{name}', name)
os.makedirs('/content/drive/MyDrive/careimpact', exist_ok=True)
shutil.copy(out, '/content/drive/MyDrive/careimpact/careimpact_ft.zip')
print('Saved to Google Drive: careimpact/careimpact_ft.zip', round(os.path.getsize(out) / 1e9, 2), 'GB')